In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
BASE_DIR = Path().resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Load files
nav_history = pd.read_csv(PROCESSED_DIR / "02_nav_history.csv", parse_dates=["date"])
fund_master = pd.read_csv(PROCESSED_DIR / "01_fund_master.csv")
benchmark = pd.read_csv(PROCESSED_DIR / "10_benchmark_indices.csv", parse_dates=["date"])

print("✅ Files loaded!")
print("Nav history shape:", nav_history.shape)

✅ Files loaded!
Nav history shape: (46000, 3)


In [2]:
# Calculate CAGR for each fund
def calculate_cagr(df):
    first_nav = df.sort_values("date").iloc[0]["nav"]
    last_nav = df.sort_values("date").iloc[-1]["nav"]
    n_trading_days = len(df)
    years = n_trading_days / 252  # 252 trading days in a year
    cagr = ((last_nav / first_nav) ** (1 / years) - 1) * 100
    return round(cagr, 2)

cagr_results = []
for code in nav_history["amfi_code"].unique():
    df = nav_history[nav_history["amfi_code"] == code]
    cagr = calculate_cagr(df)
    cagr_results.append({"amfi_code": code, "cagr_pct": cagr})

cagr_df = pd.DataFrame(cagr_results).sort_values("cagr_pct", ascending=False)
print(cagr_df.head(10))

    amfi_code  cagr_pct
12     120505     31.48
2      119598     31.10
39     149324     30.97
36     148569     30.64
34     148567     29.71
22     120843     29.65
7      100033     28.90
38     149323     28.38
26     119094     27.08
0      119551     24.77


In [3]:
# Calculate Sharpe Ratio for each fund
def calculate_sharpe(df):
    df = df.sort_values("date").copy()
    df["daily_return"] = df["nav"].pct_change()
    df = df.dropna()
    avg_return = df["daily_return"].mean()
    std_return = df["daily_return"].std()
    risk_free_rate = 0.065 / 252  # 6.5% annual risk free rate in India
    sharpe = ((avg_return - risk_free_rate) / std_return) * np.sqrt(252)
    return round(sharpe, 2)

sharpe_results = []
for code in nav_history["amfi_code"].unique():
    df = nav_history[nav_history["amfi_code"] == code]
    sharpe = calculate_sharpe(df)
    sharpe_results.append({"amfi_code": code, "sharpe_ratio": sharpe})

sharpe_df = pd.DataFrame(sharpe_results).sort_values("sharpe_ratio", ascending=False)
print(sharpe_df.head(10))

    amfi_code  sharpe_ratio
34     148567          1.45
22     120843          1.31
36     148569          1.23
0      119551          1.21
12     120505          1.18
38     149323          1.13
7      100033          1.09
15     118632          1.08
28     101206          1.03
11     120504          1.03


In [4]:
# Calculate Beta for each fund
nifty = benchmark[benchmark["index_name"] == "NIFTY50"].sort_values("date").copy()
nifty["market_return"] = nifty["close_value"].pct_change()
nifty = nifty.dropna()

beta_results = []
for code in nav_history["amfi_code"].unique():
    df = nav_history[nav_history["amfi_code"] == code].sort_values("date").copy()
    df["fund_return"] = df["nav"].pct_change()
    df = df.dropna()
    
    # merge with nifty
    merged = pd.merge(df[["date", "fund_return"]], nifty[["date", "market_return"]], on="date")
    
    if len(merged) > 10:
        covariance = merged["fund_return"].cov(merged["market_return"])
        market_variance = merged["market_return"].var()
        beta = round(covariance / market_variance, 2)
    else:
        beta = None
        
    beta_results.append({"amfi_code": code, "beta": beta})

beta_df = pd.DataFrame(beta_results).sort_values("beta", ascending=False)
print(beta_df.head(10))

    amfi_code  beta
39     149324  0.13
2      119598  0.07
13     120506  0.05
20     120841  0.04
15     118632  0.04
33     102887  0.04
28     101206  0.03
11     120504  0.02
27     119095  0.01
17     118634  0.01


In [5]:
# Calculate VaR (Value at Risk) for each fund
def calculate_var(df, confidence=0.95):
    df = df.sort_values("date").copy()
    df["daily_return"] = df["nav"].pct_change()
    df = df.dropna()
    var = np.percentile(df["daily_return"], (1 - confidence) * 100)
    return round(var * 100, 2)

var_results = []
for code in nav_history["amfi_code"].unique():
    df = nav_history[nav_history["amfi_code"] == code]
    var = calculate_var(df)
    var_results.append({"amfi_code": code, "var_95_pct": var})

var_df = pd.DataFrame(var_results).sort_values("var_95_pct")
print(var_df.head(10))

    amfi_code  var_95_pct
3      119599       -2.69
27     119095       -2.62
29     101207       -2.60
17     118634       -2.54
2      119598       -2.45
39     149324       -2.35
32     102886       -1.92
7      100033       -1.90
12     120505       -1.89
26     119094       -1.85


In [6]:
# Combine all metrics into one table
metrics_df = cagr_df.merge(sharpe_df, on="amfi_code")
metrics_df = metrics_df.merge(beta_df, on="amfi_code")
metrics_df = metrics_df.merge(var_df, on="amfi_code")

# Add scheme names
metrics_df = metrics_df.merge(fund_master[["amfi_code", "scheme_name"]], on="amfi_code", how="left")

# Reorder columns
metrics_df = metrics_df[["amfi_code", "scheme_name", "cagr_pct", "sharpe_ratio", "beta", "var_95_pct"]]

print(metrics_df.head(10))
print("\nShape:", metrics_df.shape)

# Save to processed folder
PROCESSED_DIR = BASE_DIR / "data" / "processed"
metrics_df.to_csv(PROCESSED_DIR / "performance_metrics.csv", index=False)
print("\n✅ Metrics saved!")

   amfi_code                                        scheme_name  cagr_pct  \
0     120505           ICICI Pru Midcap Fund - Regular - Growth     31.48   
1     119598         SBI Small Cap Fund - Regular Plan - Growth     31.10   
2     149324              DSP Small Cap Fund - Regular - Growth     30.97   
3     148569      Mirae Asset Tax Saver Fund - Regular - Growth     30.64   
4     148567      Mirae Asset Large Cap Fund - Regular - Growth     29.71   
5     120843             Kotak Flexicap Fund - Regular - Growth     29.65   
6     100033  HDFC Mid-Cap Opportunities Fund - Regular - Gr...     28.90   
7     149323                 DSP Midcap Fund - Regular - Growth     28.38   
8     119094                Axis Midcap Fund - Regular - Growth     27.08   
9     119551          SBI Bluechip Fund - Regular Plan - Growth     24.77   

   sharpe_ratio  beta  var_95_pct  
0          1.18 -0.02       -1.89  
1          0.95  0.07       -2.45  
2          0.95  0.13       -2.35  
3       

In [ ]:
import matplotlib.pyplot as plt

# Chart — CAGR vs Sharpe Ratio
plt.figure(figsize=(10, 6))
plt.scatter(metrics_df["cagr_pct"], metrics_df["sharpe_ratio"], color="blue", alpha=0.7)

# Add fund labels
for _, row in metrics_df.iterrows():
    plt.annotate(str(row["amfi_code"]), (row["cagr_pct"], row["sharpe_ratio"]), fontsize=6)

plt.title("CAGR vs Sharpe Ratio for All Funds")
plt.xlabel("CAGR (%)")
plt.ylabel("Sharpe Ratio")
plt.tight_layout()
plt.show()